In [2]:

from __future__ import annotations
import time
import numpy as np
from dataclasses import dataclass
from enum import Enum, auto
from typing import Protocol
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings

c:\Users\Sangram Mishra\Desktop\ADK\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [ ]:
# ─────────────────────────────────────────────────────────────────
# EMBEDDING PROVIDER ENUM
# Single source of truth for all supported providers
# Add new provider here → auto-available everywhere
# ─────────────────────────────────────────────────────────────────

class EmbedProvider(Enum):
    OLLAMA        = auto()   # Local, free, needs Ollama running
    SENTENCE_TF   = auto()   # Local, free, HuggingFace models
    OPENAI        = auto()   # Cloud, paid, best quality
    COHERE        = auto()   # Cloud, paid, multilingual
    JINA          = auto()   # Local/Cloud, long context (8192 tokens)


# ─────────────────────────────────────────────────────────────────
# EMBEDDING FACTORY
# O(1) dict dispatch — adding a provider = one dict entry
# ─────────────────────────────────────────────────────────────────

# Default model per provider — override via model_name param
_DEFAULT_MODELS: dict[EmbedProvider, str] = {
    EmbedProvider.OLLAMA:      "nomic-embed-text",
    EmbedProvider.SENTENCE_TF: "all-MiniLM-L6-v2",
    EmbedProvider.OPENAI:      "text-embedding-3-small",
    EmbedProvider.COHERE:      "embed-multilingual-v3.0",
    EmbedProvider.JINA:        "jinaai/jina-embeddings-v2-base-en",
}

print()

In [4]:
class embder(Enum):
    model1 = auto()
    model2 = auto()

In [5]:
s ={
    embder.model1
}

In [6]:
s

{<embder.model1: 1>}

In [8]:
_DEFAULT_MODELS

{<EmbedProvider.OLLAMA: 1>: 'nomic-embed-text',
 <EmbedProvider.SENTENCE_TF: 2>: 'all-MiniLM-L6-v2',
 <EmbedProvider.OPENAI: 3>: 'text-embedding-3-small',
 <EmbedProvider.COHERE: 4>: 'embed-multilingual-v3.0',
 <EmbedProvider.JINA: 5>: 'jinaai/jina-embeddings-v2-base-en'}

In [ ]:
def get_embedder(
    provider: EmbedProvider = EmbedProvider.OLLAMA,
    model_name: str | None = None,
) -> Embeddings:
    """
    Factory: returns a LangChain-compatible embedder.
    Swap provider in ONE line — rest of pipeline unchanged.

    Args:
        provider:   EmbedProvider enum value.
        model_name: Override default model for that provider.

    Returns: LangChain Embeddings object.
    O(1) — dict lookup, no if-elif chain.
    """
    model = model_name or _DEFAULT_MODELS[provider]

    # Builder functions — called only for the requested provider
    def _build_ollama() -> Embeddings:
        from langchain_ollama import OllamaEmbeddings
        return OllamaEmbeddings(model=model)

    def _build_sentence_tf() -> Embeddings:
        from langchain_community.embeddings import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name=model,
            model_kwargs={"device": "cuda"},   # auto-fallback to cpu if no GPU
            encode_kwargs={"normalize_embeddings": True},  # needed for cosine sim
        )

    def _build_openai() -> Embeddings:
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model=model)

    def _build_cohere() -> Embeddings:
        from langchain_cohere import CohereEmbeddings
        return CohereEmbeddings(model=model)

    def _build_jina() -> Embeddings:
        from langchain_community.embeddings import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name=model,
            model_kwargs={
                "device": "cuda",
                "trust_remote_code": True,     # required for Jina
            },
            encode_kwargs={"normalize_embeddings": True},
        )

    # dict dispatch — O(1)
    _builders: dict[EmbedProvider, callable] = {
        EmbedProvider.OLLAMA:      _build_ollama,
        EmbedProvider.SENTENCE_TF: _build_sentence_tf,
        EmbedProvider.OPENAI:      _build_openai,
        EmbedProvider.COHERE:      _build_cohere,
        EmbedProvider.JINA:        _build_jina,
    }

    builder = _builders.get(provider)
    if builder is None:
        raise ValueError(
            f"Unknown provider: {provider}. "
            f"Choose from: {list(_builders)}"
        )

    return builder()




In [ ]:
# ─────────────────────────────────────────────────────────────────
# EMBEDDING BENCHMARK
# Compare providers on YOUR documents
# Metrics: speed, vector quality (intra-cluster similarity)
# ─────────────────────────────────────────────────────────────────

@dataclass
class EmbedBenchmarkResult:
    provider:          str
    model:             str
    dimensions:        int
    embed_time_ms:     float          # time to embed test sentences
    avg_similarity:    float          # avg cosine sim between related pairs
    avg_dissimilarity: float          # avg cosine sim between unrelated pairs
    separation_score:  float          # higher = better (related - unrelated gap)

    def display(self) -> None:
        print(
            f"\n{'─'*55}\n"
            f"Provider     : {self.provider}\n"
            f"Model        : {self.model}\n"
            f"Dimensions   : {self.dimensions}\n"
            f"Embed time   : {self.embed_time_ms:.1f} ms\n"
            f"Related sim  : {self.avg_similarity:.4f}   "
            f"(higher = better)\n"
            f"Unrelated sim: {self.avg_dissimilarity:.4f}  "
            f"(lower = better)\n"
            f"Separation   : {self.separation_score:.4f}  "
            f"{'🏆 Excellent' if self.separation_score > 0.3 else '✅ Good' if self.separation_score > 0.15 else '⚠️ Weak'}"
        )

In [ ]:
def _cosine_similarity_matrix(vectors: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarity matrix for a batch of vectors.
    Normalise once → dot product = cosine similarity.
    O(n² × dim) — unavoidable for pairwise comparison.
    Uses numpy vectorised ops throughout.
    """
    # Normalise all vectors in one operation — O(n × dim)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    normalised = vectors / np.maximum(norms, 1e-10)

    # Matrix multiplication gives all pairwise cosine similarities
    # Shape: (n, n) — O(n² × dim) but fully vectorised
    return normalised @ normalised.T


def benchmark_embedders(
    providers: list[tuple[EmbedProvider, str | None]],
    related_pairs: list[tuple[str, str]] | None = None,
    unrelated_pairs: list[tuple[str, str]] | None = None,
) -> list[EmbedBenchmarkResult]:
    """
    Benchmark multiple embedding providers on related/unrelated sentence pairs.

    Args:
        providers:       List of (EmbedProvider, optional_model_override).
        related_pairs:   Sentence pairs that SHOULD be similar.
        unrelated_pairs: Sentence pairs that SHOULD NOT be similar.

    Returns: sorted list of EmbedBenchmarkResult (best separation first).
    """
    # Default test sentences — replace with domain-specific ones
    # for best benchmark accuracy
    if related_pairs is None:
        related_pairs = [
            ("Financial progress of SFURTI clusters",
             "Fund utilisation in traditional industry schemes"),
            ("Artisan training program in Odisha",
             "Skill development for weavers in eastern India"),
            ("Cluster implementation status report",
             "Progress update on scheme implementation"),
        ]

    if unrelated_pairs is None:
        unrelated_pairs = [
            ("Financial progress of SFURTI clusters",
             "Weather forecast for monsoon season"),
            ("Artisan training program",
             "Cricket match score update"),
            ("Cluster implementation status",
             "Recipe for dal tadka"),
        ]

    # Flatten all sentences — embed in single batch per provider
    # Avoids multiple embed() calls — more efficient
    related_flat   = [s for pair in related_pairs   for s in pair]
    unrelated_flat = [s for pair in unrelated_pairs for s in pair]
    all_sentences  = related_flat + unrelated_flat

    results: list[EmbedBenchmarkResult] = []

    for provider, model_override in providers:
        model_name = model_override or _DEFAULT_MODELS[provider]
        print(f"\n⏳ Benchmarking: {provider.name} / {model_name}")

        try:
            embedder = get_embedder(provider, model_override)

            # Single batch embed call — O(sentences × dim)
            start = time.perf_counter()
            vectors = np.array(
                embedder.embed_documents(all_sentences),
                dtype=np.float32,
            )
            elapsed_ms = (time.perf_counter() - start) * 1000

            dimensions = vectors.shape[1]
            n_related = len(related_flat)

            related_vecs   = vectors[:n_related]
            unrelated_vecs = vectors[n_related:]

            # Similarity for related pairs — O(pairs × dim)
            rel_sims: list[float] = []
            for i in range(0, len(related_vecs), 2):
                sim_mat = _cosine_similarity_matrix(related_vecs[i:i+2])
                rel_sims.append(float(sim_mat[0, 1]))

            # Similarity for unrelated pairs
            unrel_sims: list[float] = []
            for i in range(0, len(unrelated_vecs), 2):
                sim_mat = _cosine_similarity_matrix(unrelated_vecs[i:i+2])
                unrel_sims.append(float(sim_mat[0, 1]))

            avg_rel   = float(np.mean(rel_sims))
            avg_unrel = float(np.mean(unrel_sims))

            result = EmbedBenchmarkResult(
                provider=          provider.name,
                model=             model_name,
                dimensions=        dimensions,
                embed_time_ms=     elapsed_ms,
                avg_similarity=    avg_rel,
                avg_dissimilarity= avg_unrel,
                separation_score=  avg_rel - avg_unrel,  # key metric
            )
            results.append(result)
            result.display()

        except Exception as e:
            print(f"  ❌ Failed: {e}")

    # Sort by separation score descending — best model first
    results.sort(key=lambda r: r.separation_score, reverse=True)

    print(f"\n{'═'*55}")
    print(f"🏆 Best for your documents: {results[0].provider} / {results[0].model}")
    print(f"   Separation score: {results[0].separation_score:.4f}")

    return results


# ─────────────────────────────────────────────────────────────────
# EMBEDDING CACHE
# Avoid re-embedding same text twice
# dict: O(1) lookup, O(1) insert
# ─────────────────────────────────────────────────────────────────


In [ ]:
class CachedEmbedder:
    """
    Wraps any LangChain embedder with an in-memory cache.
    Cache key: text string (immutable, hashable)
    Cache value: embedding vector (list of floats)

    Why: embedding is the slowest step per chunk.
         Same chunk text appearing in multiple docs
         only gets embedded once.

    Tradeoff: RAM usage grows with unique chunks.
              For >100k chunks, use a persistent cache (Redis/SQLite).
    """

    def __init__(self, embedder: Embeddings) -> None:
        self._embedder = embedder
        # dict: O(1) average lookup and insert
        self._cache: dict[str, list[float]] = {}
        self._hits   = 0
        self._misses = 0

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """
        Embed batch — only call embedder for cache misses.
        O(n) scan, O(1) per lookup.
        """
        results: list[list[float] | None] = [None] * len(texts)
        miss_indices: list[int] = []
        miss_texts:   list[str] = []

        # Partition into hits and misses — O(n)
        for i, text in enumerate(texts):
            cached = self._cache.get(text)      # O(1)
            if cached is not None:
                results[i] = cached
                self._hits += 1
            else:
                miss_indices.append(i)
                miss_texts.append(text)
                self._misses += 1

        # Batch embed only misses — single API/model call
        if miss_texts:
            new_embeddings = self._embedder.embed_documents(miss_texts)
            for idx, text, embedding in zip(
                miss_indices, miss_texts, new_embeddings
            ):
                self._cache[text] = embedding   # O(1) insert
                results[idx] = embedding

        return results  # type: ignore[return-value]

    def embed_query(self, text: str) -> list[float]:
        """Single query embed — check cache first."""
        if text in self._cache:                 # O(1)
            self._hits += 1
            return self._cache[text]

        self._misses += 1
        embedding = self._embedder.embed_query(text)
        self._cache[text] = embedding
        return embedding

    @property
    def cache_stats(self) -> dict[str, int | float]:
        total = self._hits + self._misses
        return {
            "hits":      self._hits,
            "misses":    self._misses,
            "total":     total,
            "hit_rate":  round(self._hits / total, 3) if total > 0 else 0.0,
            "cached":    len(self._cache),
        }